In [1]:
import json

from pathlib import Path

import pandas as pd
import scanpy as sc
import numpy as np

from dotenv import dotenv_values

from project_utils import FileTracker

In [3]:
# Load config file agnostic to project or notebook work directory
config_path = Path("config.env")
if not config_path.exists():
    config_path = Path("../../config.env")
CONFIG = dotenv_values(config_path)
assert CONFIG != {}, 'Configuration is empty - check the filepath.'

DATADIR = Path(CONFIG['DATADIR'])
OUTDIR = DATADIR / "9_objects" 

tracker = FileTracker(log_file=DATADIR/'tracked_files.toml')

In [3]:
# Load full AnnData
scdata = sc.read_h5ad(
    OUTDIR / 'singlecell/hypomap3d_singlecell_all.h5ad'
)

In [4]:
# Load region relmass table
region_relmass = pd.read_csv(OUTDIR / 'spatial/regionwise_relative_masses/regionwise_relative_masses_wo_fibers.csv', index_col=0).drop(
    columns=['celltype_descr', 'celltype_abc_like', 'flag']
)

In [5]:
# Load structure tree
with open( OUTDIR / 'spatial/ABC_structure_tree_processed.json') as fp:
    structure_tree = json.load(fp)

### Quick alterations on .obs

In [7]:
scdata.obs.columns

Index(['CellID', 'Dataset', 'Sample', 'total_counts', 'n_genes', 'pct_mito',
       'Author', 'Source', 'Data_source_link', 'GEO', 'Publication_Link',
       'Suspension', 'Technology', 'Author_Sex_harmonized', 'Sex', 'Strain',
       'Diet', 'Treatment', 'Age', 'Mean_age_in_weeks', 'IsControl', 'L1_ID',
       'L2_ID', 'L3_ID', 'L4_ID', 'L5_ID', 'L1_descr', 'L2_descr', 'L3_descr',
       'L4_descr', 'L5_descr', 'L1_abc_like', 'L2_abc_like', 'L3_abc_like',
       'L4_abc_like', 'L5_abc_like'],
      dtype='object')

In [8]:
scdata.obs = scdata.obs.rename(
    columns={
        'sex_per_sample_inferred': 'Sex'     
    }
)

In [11]:
scdata.obs = scdata.obs[
    [
        'Author',
        'Dataset',
        'Sample',
        'IsControl',
        'L1_ID',
        'L2_ID',
        'L3_ID',
        'L4_ID',
        'L5_ID',
        'L1_descr',
        'L2_descr',
        'L3_descr',
        'L4_descr',
        'L5_descr',
        'L1_abc_like',
        'L2_abc_like',
        'L3_abc_like',
        'L4_abc_like',
        'L5_abc_like',
        'Sex',
        'Suspension',
        'Technology',
        'total_counts',
        'n_genes',
        'pct_mito'
    ]
].copy()

### Addition of Region columns

In [12]:
# Top assignments (taken from name)

scdata.obs['top_region'] = [
    cat.split()[1] if not cat.startswith('NN') else None
    for cat in scdata.obs['L3_descr'].values
]
scdata.obs['top_region'].astype('category')

# Umbrella region

UMBRELLA_REGION_MAP = {
    "Mammillary regions": ["MMl", "MMm", "MMme","TMv", "PMv", "SUM", "LM", "PMd"],
    "PVH region": ["PVH", "PVHd", "PVH-SO"],
    "Basal HY": ["ARH", "VMH", "TU"],
    "Dorsal-lateral HY": ["LHA", "DMH"],
    "Anterior HY": ["AHN", "SBPV", "PVi",],
    "Preoptic HY": ["LPO", "MEPO", "MPO", "MPN", "PS", "PVpo", "AVPV" ],
    "ZI/Subthalamic": ["PSTN", "STN", "ZI"],
    "Extra-hypothalamic": ["BST", "CP", "LSr", "LSv", "MB", "MEA", "NDB", "RT", "VAL", "VPM", "SI", "MS"],
    "Posterior HY": ["PH", "PVp", ],
    "HY (Gnrh1 neurons)": ["HY"]
}

REGION_UMBRELLA_MAP = {
    region: umbrella
    for umbrella, regions in UMBRELLA_REGION_MAP.items()
    for region in regions
}

scdata.obs['umbrella_region'] = scdata.obs['top_region'].map(REGION_UMBRELLA_MAP)


In [13]:
acronym_id_map = {
    v['acronym']: k
    for k, v in structure_tree.items()
}

# Identify which columns belong to HY
columns_ids = [acronym_id_map[a] for a in region_relmass.columns]
columns_mask = [
    1097 in structure_tree[id_]['path']
    for id_ in columns_ids
]

# Keep HY subregions, collapse all non-HY regions
region_relmass_collapsed = region_relmass.loc[:, columns_mask].copy()
region_relmass_collapsed['non-HY'] = region_relmass.loc[:, ~np.array(columns_mask)].sum(axis=1)

# Drop regions with very low max contribution (small kinda irrelevant)
mask_filter = region_relmass_collapsed.max(axis=0) > 0.1
region_relmass_collapsed = region_relmass_collapsed.loc[:, mask_filter]

# Rename columns and add to .obs
for col in region_relmass_collapsed.columns:
    new_name = f"{col}_relmass"
    
    relmass_map = region_relmass_collapsed[col]
    
    scdata.obs[new_name] = scdata.obs['L3_ID'].map(relmass_map).astype(np.float32)

### Final tidy up

In [20]:
sc.read_h5ad('/REDACTED-path/HypoMap3D/9_Objects/v0.6/singlecell/sc_anndata_all_cellxgene_optimized.h5ad', backed='r') # old object for reference

AnnData object with n_obs × n_vars = 1574305 × 56939 backed at '/REDACTED-path/HypoMap3D/9_Objects/v0.6/singlecell/sc_anndata_all_cellxgene_optimized.h5ad'
    obs: 'IsControl', 'L1_descr', 'L2_descr', 'L3_descr', 'L4_descr', 'L5_descr', 'L1_abc_like', 'L2_abc_like', 'L3_abc_like', 'L4_abc_like', 'L5_abc_like', 'Dataset', 'Sample', 'Author', 'Suspension', 'Technology', 'Sex_inferred', 'total_counts', 'n_genes', 'pct_mito'
    obsm: 'X_umap'

In [22]:
# Clean irrelevant .obsm etc
del scdata.obsm['X_embed']

In [24]:
# Normalize the data
sc.pp.normalize_total(scdata, target_sum=10_000)
sc.pp.log1p(scdata)

In [ ]:
# Log1p uns slot incompatible with cellxgene – will crash it
scdata.uns.clear()

### Conversion to CSC Matrix

In [31]:
scdata.X = scdata.X.tocsc()

In [36]:
if (scdata.X.indices.max() < 2**31) and (scdata.X.indices.dtype != 'int32'):
    print("Applying indices optimization")
    scdata.X.indices = scdata.X.indices.astype(np.int32)

Applying indices optimization


In [45]:
scdata.write_h5ad(OUTDIR / 'singlecell/cellxgene_hm3d_v0.9_sc_all.h5ad')